In [ ]:
# Parameters
UF = "BRASIL"


## Seção 4.1.1 Visualização das séries temporais de cada estação

Este notebook reproduz a tabela 20 e a figura 32 da seção 4.1.1 do Relatório Anual de Qualidade do Ar

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


### Tabela 20 - Identificação das estações de monitoramento incluídas na base de dados desenvolvida neste relatório.

O código gera uma tabela interativa contendo as estações de monitoramento da qualidade do ar incluídas na base de dados do relatório.

A tabela permite:

- Visualizar as estações cadastradas;
- Filtrar por UF;
- Filtrar por poluentes monitorados;
- Filtrar se a estação foi incluída ou não na base de dados consolidada;
- Pesquisar registros específicos.

In [5]:
# Importação de bibliotecas necessárias 
import os
import folium
import pandas as pd
import base64
import numpy as np
import warnings
import importlib

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import secao_3.flagTables as flagtab
importlib.reload(flagtab)
warnings.filterwarnings('ignore')

aqmData = pd.read_csv('https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv')
path_bandeiras = 'https://arquivos.lcqar.ufsc.br/rqar-national-guide-files/_static/bandeiras/'

In [ ]:
# Arrumando conteúdo das colunas da tabela Monitoramento_QAr_BR.csv
aqmData['STATUS'] = aqmData['STATUS'].replace(np.nan, 'Desconhecido', regex=True)
aqmData['CATEGORIA'] = aqmData['CATEGORIA'].replace(np.nan, 'Desconhecida', regex=True)
aqmData.loc[aqmData['ID_OEMA'].isna(), 'ID_OEMA'] = '-'
aqmData.loc[aqmData['CIDADE'].isna(), 'CIDADE'] = '-'
aqmData.loc[aqmData['ID_MMA_COMPLETO'].isna(), 'ID_MMA_COMPLETO'] = '-'
aqmData.loc[aqmData['BASE_DADOS'] == False, 'BASE_DADOS'] = 'Não'
aqmData.loc[aqmData['BASE_DADOS'] == True,  'BASE_DADOS'] = 'Sim'


# Criação da tabela de bandeiras
columnsSelector = ["", "UF", "ID_OEMA", "Status", "Poluente", "Base de dados"]
aqmData_tbl = flagtab.flagTable3(aqmData, columnsSelector, path_bandeiras)
searchPaneColumns = [1, 4, 5]


# Filtra linhas com UF não declarada (ou vazia/nula)
uf_series = aqmData_tbl['UF'].astype(str).str.strip()
mask_invalid_uf = (
    uf_series.eq('') |
    uf_series.str.lower().isin(['não declarado', 'nao declarado', 'nan']) |
    aqmData_tbl['UF'].isna()
)
aqmData_tbl = aqmData_tbl[~mask_invalid_uf].copy()


# Ajusta coluna "Base de dados" para texto Sim/Não
aqmData_tbl.loc[aqmData_tbl['Base de dados'] == False, 'Base de dados'] = 'Não'
aqmData_tbl.loc[aqmData_tbl['Base de dados'] == True,  'Base de dados'] = 'Sim'


# Gera tabela interativa final
html_tabela20 = flagtab.tabela_iterativa(aqmData_tbl, searchPaneColumns)

In [ ]:
# Salvar a tabela interativo como um arquivo HTML e abrir no navegador
import webbrowser
import os

output_dir = "outputs"
output_path = os.path.join(output_dir, "tabela20.html")

os.makedirs(output_dir, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_tabela20)

webbrowser.open(output_path)

### Figura 32 - Disposição espacial das estações de monitoramento da qualidade do ar.

Este script gera um mapa interativo da Rede Nacional de Monitoramento da Qualidade do Ar.

Cada estação de monitoramento é representada por um marcador geográfico. Ao clicar sobre uma estação, o usuário pode visualizar:

- Identificação da estação;
- Município;
- Categoria da estação;
- Status operacional;
- Links para gráficos de séries temporais dos poluentes monitorados.

Esses gráficos permitem analisar:
- Série temporal bruta; 
- Médias diárias; 
- Médias mensais; 
- Variação por hora do dia; 
- Variação por dia da semana.

In [ ]:
# -*- coding: utf-8 -*-
"""
Mapa Folium - Rede Nacional de Monitoramento da Qualidade do Ar
Correções:
- Remove pontos com coordenadas NaN
- Corrige warning do groupby.apply
- Garante links de todas as séries temporais por estação
"""

import os
import pandas as pd
import numpy as np
import folium
from folium.plugins import MiniMap
import requests

# Mapeamento sigla -> código IBGE (necessário para a API de malhas)
UF_CODES = {
    "AC": 12, "AL": 27, "AP": 16, "AM": 13, "BA": 29, "CE": 23, "DF": 53,
    "ES": 32, "GO": 52, "MA": 21, "MT": 51, "MS": 50, "MG": 31, "PA": 15,
    "PB": 25, "PR": 41, "PE": 26, "PI": 22, "RJ": 33, "RN": 24, "RS": 43,
    "RO": 11, "RR": 14, "SC": 42, "SP": 35, "SE": 28, "TO": 17,
}
# "BRASIL" (ou None/""/"BR"/"TODOS") => sem filtro, mapa do país inteiro.
uf_filter = None if UF in (None, "", "BRASIL", "BR", "TODOS") else UF


# Pré-processamento
aqmData['STATUS'] = aqmData['STATUS'].replace(np.nan, 'Desconhecido', regex=True)
aqmData['CATEGORIA'] = aqmData['CATEGORIA'].replace(np.nan, 'Desconhecida', regex=True)
aqmData['ID_MMA'] = aqmData['ID_MMA'].fillna('-')
aqmData['ID_MMA_COMPLETO'] = aqmData['ID_MMA_COMPLETO'].fillna('-')
aqmData['ID_OEMA'] = aqmData['ID_OEMA'].fillna('-')
aqmData['CIDADE'] = aqmData['CIDADE'].fillna('-')

aqmData['LATITUDE'] = pd.to_numeric(aqmData['LATITUDE'].astype(str).str.replace(',', '.'), errors='coerce')
aqmData['LONGITUDE'] = pd.to_numeric(aqmData['LONGITUDE'].astype(str).str.replace(',', '.'), errors='coerce')


# 2. Remove linhas onde a conversão falhou (NaN)
aqmData = aqmData.dropna(subset=['LATITUDE', 'LONGITUDE'])

# 3. Agora o isfinite funcionará corretamente
aqmData = aqmData[np.isfinite(aqmData['LATITUDE']) & np.isfinite(aqmData['LONGITUDE'])]
if uf_filter is not None:
    aqmData = aqmData[aqmData["UF"] == uf_filter]


# Função auxiliar
def safe_initial(val):
    if pd.isna(val) or not str(val).strip():
        return '-'
    return str(val).strip()[0].upper()


# Criação da coluna de categoria/status
aqmData['status_category'] = (
    aqmData['STATUS'].apply(safe_initial)
    + aqmData['CATEGORIA'].apply(safe_initial)
)

color_map = {
    'AR': '#038cfc',   # Referência/Ativa
    'IR': '#c8f2fa',   # Referência/Inativa
    'AI': '#f08432',   # Indicativa/Ativa
    'II': '#ffea8f'    # Indicativa/Inativa
}


# Agrupamento preservando correspondência 1:1

cols_group = ['UF', 'CIDADE', 'LATITUDE', 'LONGITUDE', 'STATUS', 'CATEGORIA', 'ID_OEMA']

def merge_station(df):
    """Agrupa por estação mantendo correspondência entre poluente, base_dados e ID"""
    registros = []
    for _, row in df.iterrows():
        registros.append({
            'POLUENTE': str(row['POLUENTE']).strip(),
            'ID_MMA_COMPLETO': str(row['ID_MMA_COMPLETO']).strip(),
            'BASE_DADOS': str(row['BASE_DADOS']).strip().lower() in ['true', '1']
        })
    return pd.Series({
        'ID_MMA': df['ID_MMA'].mode()[0] if not df['ID_MMA'].mode().empty else '-',
        'LINKS': registros
    })


# evita o FutureWarning
aqmDataGrouped = (
    aqmData.groupby(cols_group, dropna=False)
    .apply(merge_station, include_groups=False)
    .reset_index()
)


# Recria status_category
aqmDataGrouped['status_category'] = (
    aqmDataGrouped['STATUS'].apply(safe_initial)
    + aqmDataGrouped['CATEGORIA'].apply(safe_initial)
)


# Mapa base
brazil_bounds = [[-33.75, -73.98], [5.27, -34.79]]
map_clusters = folium.Map(
    location=[-15, -55],
    tiles="OpenStreetMap",
    zoom_start=5,
    min_zoom=4,
    max_bounds=True
)
if uf_filter is not None:
    # Fronteira do estado (igual ao Estadual): busca a malha na API do IBGE
    # e ajusta o zoom, com fallback para o bounding box dos pontos filtrados.
    uf_code = UF_CODES.get(str(uf_filter).upper())
    boundary_geojson = None
    if uf_code is not None:
        boundary_url = (
            f"https://servicodados.ibge.gov.br/api/v2/malhas/{uf_code}"
            f"?resolucao=2&formato=application/vnd.geo+json"
        )
        resp = requests.get(boundary_url)
        boundary_geojson = resp.json() if resp.ok else None

    if boundary_geojson is not None:
        boundary_layer = folium.GeoJson(
            boundary_geojson,
            name=f"Fronteira {uf_filter}",
            style_function=lambda feat: {"fillOpacity": 0, "color": "#000000", "weight": 2},
        )
        boundary_layer.add_to(map_clusters)
        map_clusters.fit_bounds(boundary_layer.get_bounds())
    else:
        pts_bounds = [
            [aqmDataGrouped['LATITUDE'].min(), aqmDataGrouped['LONGITUDE'].min()],
            [aqmDataGrouped['LATITUDE'].max(), aqmDataGrouped['LONGITUDE'].max()],
        ]
        map_clusters.fit_bounds(pts_bounds)
else:
    map_clusters.fit_bounds(brazil_bounds)
MiniMap(toggle_display=True).add_to(map_clusters)


# Marcadores no mapa
for _, row in aqmDataGrouped.iterrows():
    lat, lon = row['LATITUDE'], row['LONGITUDE']
    if pd.isna(lat) or pd.isna(lon):
        continue

    popup_html = f"""
        <b>ID_MMA:</b> {row.get('ID_MMA', '-')}<br>
        <b>ID_OEMA:</b> {row.get('ID_OEMA', '-')}<br>
        <b>Cidade:</b> {row.get('CIDADE', '-')}<br>
        <b>Categoria:</b> {row.get('CATEGORIA', '-')}<br>
        <b>Status:</b> {row.get('STATUS', '-')}<br>
    """

    popup_pol = ''
    for item in row.get('LINKS', []):
        pol = item.get('POLUENTE', '')
        id_val = item.get('ID_MMA_COMPLETO', '')
        base_flag = item.get('BASE_DADOS', False)
        if base_flag and id_val and id_val != '-':
            popup_pol += (
                f'<br><a href="../_static/plotly_figures/timeSeriesFigures/{id_val}.html" '
                f'target="_blank">Série temporal de {pol}</a>'
            )
        else:
            popup_pol += f'<br><i>{pol} — sem série temporal</i>'

    popup = folium.Popup(popup_html + popup_pol, max_width=800, max_height=400)

    folium.CircleMarker(
        location=[float(lat), float(lon)],
        radius=4,
        fill=True,
        fill_opacity=0.7,
        popup=popup,
        color=color_map.get(row['status_category'], 'gray'),
        fill_color=color_map.get(row['status_category'], 'gray')
    ).add_to(map_clusters)


# Legenda
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 180px;
    background-color: white;
    border:2px solid grey;
    z-index:9999;
    font-size:14px;
    padding: 10px;
">
<b>Legenda</b><br>
<i style="background:#038cfc; width:10px; height:10px; float:left; margin-right:5px;"></i> Referência/Ativa<br>
<i style="background:#c8f2fa; width:10px; height:10px; float:left; margin-right:5px;"></i> Referência/Inativa<br>
<i style="background:#f08432; width:10px; height:10px; float:left; margin-right:5px;"></i> Indicativa/Ativa<br>
<i style="background:#ffea8f; width:10px; height:10px; float:left; margin-right:5px;"></i> Indicativa/Inativa<br>
</div>
"""
map_clusters.get_root().html.add_child(folium.Element(legend_html))

map_clusters

### Código que gera os htmls necessários para o código acima

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Wed Sep 11 10:45:33 2024

@author: leohoinaski
"""

#-----------------------------Importação de pacotes ------------------------------------


import os
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


def tratar_dados(df):
    """
    Recebe DataFrame cru, cria coluna datetime, converte e limpa valores.
    Retorna DataFrame com índice datetime e coluna 'Valor' em float, valores < 0 viram NaN.

    Parameters
    ----------
    df : TYPE
        DF contendo dados brutos com colunas, sem coluna de datetime .

    Returns
    -------
    df : TYPE
        DataFrame tratado com indice de datetime e valores numéricos prontos para análise.

    """
    

    # Substitui , por . 
    df['VALOR'] = df['VALOR'].replace(',', '.', regex=True).copy()
    # Converte para float, forçando erro para NaN
    df['VALOR'] = pd.to_numeric(df['VALOR'], errors='coerce').copy()
    # Transforma valores negativos em NaN
    df.loc[df['VALOR'] < 0, 'VALOR'] = np.nan
    df['DATETIME'] = pd.to_datetime(df['DATETIME'], errors='coerce', infer_datetime_format=True)
    time_range = pd.date_range(df['DATETIME'].min(), df['DATETIME'].max(), freq='h').to_series(name='DATETIME')
    df = pd.merge(time_range, df,how='left')
    #df = df.set_index('datetime', drop=False)
    df['DATETIME'] = pd.to_datetime(df['DATETIME']).copy()
    return df

def split_nan_segments(x, y):
    """Splits x and y into segments where y is not NaN"""
    segments = []
    x = np.array(x)
    y = np.array(y)
    
    isnan = np.isnan(y)
    start = 0
    for i in range(1, len(y)):
        if isnan[i] and not isnan[i-1]:
            segments.append((x[start:i], y[start:i]))
            start = i + 1
        elif not isnan[i] and isnan[i-1]:
            start = i
    if not isnan[-1]:
        segments.append((x[start:], y[start:]))
    return segments

    
def iterative_timeseries(df,row):
    df = tratar_dados(df)
    daily_avg_df = df[['DATETIME','VALOR']].groupby(pd.Grouper(key='DATETIME', freq='D')).quantile(0.50)
    daily_min_df = df[['DATETIME','VALOR']].groupby(pd.Grouper(key='DATETIME', freq='D')).quantile(0.05)
    daily_max_df = df[['DATETIME','VALOR']].groupby(pd.Grouper(key='DATETIME', freq='D')).quantile(0.95)
        
    month_avg_df = df.groupby("MES")["VALOR"].quantile(0.50).reset_index()
    month_min_df = df.groupby("MES")["VALOR"].quantile(0.05).reset_index()
    month_max_df = df.groupby("MES")["VALOR"].quantile(0.95).reset_index()
    
    hourly_min = df.groupby('HORA')[['VALOR']].quantile(0.05)
    hourly_max = df.groupby('HORA')[['VALOR']].quantile(0.95)
    hourly_average = df.groupby('HORA')[['VALOR']].quantile(0.50)

    df['day_of_week_name'] = df['DATETIME'].dt.day_name()
    min_by_day_name = df.groupby('day_of_week_name')[['VALOR']].quantile(0.05)
    max_by_day_name = df.groupby('day_of_week_name')[['VALOR']].quantile(0.95)
    average_by_day_name = df.groupby('day_of_week_name')[['VALOR']].quantile(0.50)
    
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    min_by_day_name = min_by_day_name.reindex(day_order)
    max_by_day_name = max_by_day_name.reindex(day_order)
    average_by_day_name = average_by_day_name.reindex(day_order)
    
    dates = daily_min_df.index

    # Create the figure
    #fig = go.Figure()
    fig = make_subplots(rows=5, cols=1)
    
    # raw
    fig.add_trace(go.Scatter(
        x=df.DATETIME,
        y=df.VALOR,
        mode='lines',
        line=dict(color='rgba(100, 100, 100, 0.5)', width=1),
        name='Séries bruta'
    ), row=5, col=1)
    
    
    # daily 
    segments = split_nan_segments(dates, daily_max_df.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(255, 204, 204,.4)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            fillcolor='rgba(255, 204, 204,.4)',
            #name='Máximo',
            showlegend=False,
            connectgaps=False
        ), row=4, col=1)
    
    fig.add_trace(go.Scatter(
        x=dates,
        y=daily_avg_df.VALOR,
        mode='lines',
        line=dict(color='rgba(255, 0, 0, 1)', width=1), # Red line with 50% opacity
        #name='Média'
        showlegend=False,
    ), row=4, col=1)

    segments = split_nan_segments(dates, daily_min_df.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(255, 204, 204,0.4)', width=2),
            fillcolor='rgba(255,255,255,1)',
            #name='Mínimo',
            showlegend=False,
            connectgaps=False
        ), row=4, col=1)
    

    # montly 
    segments = split_nan_segments(month_max_df.index, month_max_df.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(255, 204, 153, 0.2)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            fillcolor='rgba(255, 204, 153, 0.2)',
            #name='Máximo',
            mode='lines',
            showlegend=False,
            connectgaps=False
        ), row=3, col=1)

    fig.add_trace(go.Scatter(
        x=month_avg_df.index,
        y=month_avg_df.VALOR,
        mode='lines',
        showlegend=False,
        line=dict(color='rgba(255, 165, 0, 1)', width=2), # Red line with 50% opacity
        #name='Média'
    ), row=3, col=1)
    
    segments = split_nan_segments(month_min_df.index, month_min_df.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(255, 204, 153, 0.2)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            fillcolor='rgba(255,255,255,1)',
            #name='Mínimo',
            mode='lines',
            showlegend=False,
            connectgaps=False
        ), row=3, col=1)
    
    
    # week 
    segments = split_nan_segments(max_by_day_name.index, max_by_day_name.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(153, 204, 255, 0.1)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            fillcolor='rgba(153, 204, 255, 0.1)',
            #name='Máximo',
            mode='lines',
            showlegend=False,
            connectgaps=False
        ), row=2, col=1)
    
    fig.add_trace(go.Scatter(
        x=average_by_day_name.index,
        y=average_by_day_name.VALOR,
        mode='lines',
        showlegend=False,
        line=dict(color='rgba(153, 204, 255, 1)', width=2), # Red line with 50% opacity
        #name='Média'
    ), row=2, col=1)

    segments = split_nan_segments(min_by_day_name.index, min_by_day_name.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(153, 204, 255, 0.1)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            fillcolor='rgba(255,255,255,1)',
            showlegend=False,
            #name='Mínimo',
            mode='lines',
            connectgaps=False
        ), row=2, col=1)

    
    # hourly 
        
    segments = split_nan_segments(hourly_max.index, hourly_max.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(153, 153, 255, 0.1)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            fillcolor='rgba(153, 153, 255,0.1)',
            showlegend=False,
            #name='Máximo',
            mode='lines',
            connectgaps=False
        ), row=1, col=1)
    

    
    fig.add_trace(go.Scatter(
        x=hourly_average.index,
        y=hourly_average.VALOR,
        mode='lines',
        showlegend=False,
        line=dict(color='rgba(153, 153, 255, 1)', width=2), # Red line with 50% opacity
        #name='Média'
    ), row=1, col=1)


    segments = split_nan_segments(hourly_min.index, hourly_min.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            fill='tozeroy',
            line=dict(color='rgba(153, 153, 255, 0.1)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            fillcolor='rgba(255,255,255,1)',
            showlegend=False,
            #name='Mínimo',
            mode='lines',
            connectgaps=False
        ), row=1, col=1)


    # Update layout for better presentation
    fig.update_layout(
        title=dict(text='ID_MMA: '+str(row.ID_MMA_COMPLETO)+ ' - ID_OEMA: '+str(row.ID_OEMA)+' - Cidade: '+str(row.CIDADE)+
                   '<br> Poluente:'+str(row.POLUENTE) +
                   '<br> Início: '+str(row.INICIO) + '   Fim: '+str(row.FIM)+
                   '<br> Faixa =  5°-95° percentil - Linha = 50° percentil'),
            font=dict(
                size=8,  # Set the desired font size here
                family="Arial",
                color="black"),
        hovermode='x unified', # Shows hover info for all traces at a given x-coordinate
        height=1200, width=800,
        plot_bgcolor='rgba(0.9,0.9,0.9,0.2)',
        showlegend=False)

    #unidade = df.loc[0,"UNIDADE"]
    unidade = '(ug/m³)'
    fig.update_yaxes(title_text="Concentração<br>Médias nas horas<br>"+unidade, row=1, col=1)
    fig.update_xaxes(title_text="Hora", row=1, col=1)

    fig.update_yaxes(title_text="Concentração<br>Médias nos dias da semana<br>"+unidade, row=2, col=1)
    fig.update_xaxes(title_text="Dia da semana", row=2, col=1)

    fig.update_yaxes(title_text="Concentração<br>Médias mensais<br>"+unidade, row=3, col=1)
    fig.update_xaxes(title_text="Mês/Ano", row=3, col=1)

    fig.update_yaxes(title_text="Concentração<br>Médias diárias<br>"+unidade, row=4, col=1)
    fig.update_xaxes(title_text="Dia/Ano", row=4, col=1)

    fig.update_yaxes(title_text="Concentração<br>Série completa"+unidade, row=5, col=1)
    fig.update_xaxes(title_text="Dia/Mês/Ano Hora", row=5, col=1)

    rootPath = os.path.dirname(os.getcwd())
    
    fig.write_html(rootPath+"/_static/plotly_figures/timeSeriesFigures/"+row.ID_MMA_COMPLETO+".html")

    
    return fig


def iterative_raw_timeseries(df):
    df = tratar_dados(df)
    dates = df['DATETIME']

    # Create the figure
    #fig = go.Figure()
    fig = make_subplots(rows=1, cols=1)
    
    # raw

    segments = split_nan_segments(df['DATETIME'], df.VALOR)
    for seg_x, seg_y in segments:
        fig.add_trace(go.Scatter(
            x=seg_x,
            y=seg_y,
            #fill='tozeroy',
            line=dict(color='rgba(153, 153, 255, 1)', width=1),
            #fillcolor='rgba(255,255,255,1)',
            #fillcolor='rgba(153, 153, 255,0.1)',
            showlegend=False,
            name='Série temporal',
            mode='lines',
            connectgaps=False
        ))
    

    # Update layout for better presentation
    fig.update_layout(
        title='Série temporal',
        hovermode='x unified', # Shows hover info for all traces at a given x-coordinate
        height=600, width=800,
        plot_bgcolor='rgba(0.9,0.9,0.9,0.2)')

    unidade = '(ug/m³)'

    fig.update_yaxes(title_text="Concentração<br>Série completa"+unidade, row=5, col=1)
    fig.update_xaxes(title_text="Dia/Mês/Ano Hora")

    #rootPath = os.path.dirname(os.getcwd())
    
    #fig.write_html(rootPath+"/data/MQAr/plotly_figures/stationA_PM25.html")

    
    return fig

In [ ]:
# Gera os HTMLs de série temporal por estação/poluente (necessário para os links do
# mapa acima funcionarem). Só precisa ser executado uma vez -- ou quando os dados de
# origem forem atualizados; reexecuções pulam arquivos já gerados.
TIMESERIES_BASE_URL = "https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr/"

rootPath = os.path.dirname(os.getcwd())
output_dir = os.path.join(rootPath, "_static", "plotly_figures", "timeSeriesFigures")
os.makedirs(output_dir, exist_ok=True)

rows_com_dados = aqmData[aqmData['BASE_DADOS'] == True]
print(f"Gerando série temporal para {len(rows_com_dados)} combinações estação/poluente...")

geradas, puladas, erros = 0, 0, 0
for _, row in rows_com_dados.iterrows():
    pol = str(row['POLUENTE']).strip()
    id_val = str(row['ID_MMA_COMPLETO']).strip()
    if not pol or not id_val or id_val == '-':
        erros += 1
        continue

    output_path = os.path.join(output_dir, f"{id_val}.html")
    if os.path.exists(output_path):
        puladas += 1
        continue

    url = f"{TIMESERIES_BASE_URL}{pol}/{id_val}.csv"
    try:
        df_station = pd.read_csv(url)
    except Exception as e:
        print(f"Erro ao carregar {url}: {e}")
        erros += 1
        continue

    try:
        iterative_timeseries(df_station, row)
        geradas += 1
    except Exception as e:
        print(f"Erro ao gerar HTML para {id_val}: {e}")
        erros += 1

print(f"Concluído. Geradas: {geradas} | Já existentes (puladas): {puladas} | Erros: {erros}")